# Delhi/NCR Rental Market Intelligence - Data Cleaning Pipeline
This notebook documents the cleaning pipeline used to transform the messy raw rental listing data into a structured, validated, and normalized dataset for analysis.
### Objectives:
1. Normalizing pricing data (monthly rent, deposit, maintenance, electricity estimate)
2. Normalizing sizes (area in sqft)
3. Normalizing distance metrics
4. Resolving inconsistent text cases and abbreviations in location and furnishing fields
5. Handling duplicate listings and resolving null values
6. Outlier removal (erroneous data points)


In [1]:
import pandas as pd
import numpy as np
import re

# Load messy raw data
df_raw = pd.read_csv('../data/raw/rental_listings_raw.csv')
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head(3)


Raw dataset shape: (850, 43)


### 1. Cleaning Locality Text & Standardizing Casing
Locality names are loaded with varying casings (lower, upper) and abbreviations like 'Sec 62' instead of 'Sector 62'. We standardise these names.


In [1]:
# Strip whitespace, set to Title Case
df = df_raw.copy()
df['locality'] = df['locality'].astype(str).str.strip().str.title()

# Normalize sector notations
df['locality'] = df['locality'].str.replace(r'\bSec\b', 'Sector', regex=True)
df['locality'] = df['locality'].str.replace(r'\bSec-\b', 'Sector ', regex=True)
df['locality'] = df['locality'].str.replace(r'\bSector\s*-\s*', 'Sector ', regex=True)
df['locality'] = df['locality'].str.replace(r'\s+', ' ', regex=True)

print(f"Unique localities after cleaning: {df['locality'].nunique()}")
print(df['locality'].unique()[:10])


Unique localities after cleaning: 31
['Mukherjee Nagar' 'Model Town' 'Kamla Nagar' 'Civil Lines' 'Rohini'
 'Karol Bagh' 'Patel Nagar' 'Rajinder Nagar' 'Janakpuri' 'Laxmi Nagar']


### 2. Normalizing Pricing Columns
We normalize fields like '₹15,000', '15k', and 'Rs. 15,000/month' to standard integer values.


In [1]:
def clean_number(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    s = str(val).strip().replace('₹', '').replace('Rs.', '').replace('/month', '').replace(',', '').strip()
    if s.lower().endswith('k'):
        try:
            return int(float(s[:-1]) * 1000)
        except:
            return None
    try:
        return int(float(s))
    except:
        return None

df['monthly_rent'] = df['monthly_rent'].apply(clean_number)
df['security_deposit'] = df['security_deposit'].apply(clean_number)
df['maintenance'] = df['maintenance'].apply(clean_number)
df['electricity_estimate'] = df['electricity_estimate'].apply(clean_number)

df[['monthly_rent', 'security_deposit', 'maintenance', 'electricity_estimate']].describe()


### 3. Normalizing Sizes & Furnishing Status
Standardize area square footage and map furnishing to: Unfurnished, Semi-Furnished, or Furnished.


In [1]:
def clean_area(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    s = str(val).lower().replace('sq.ft.', '').replace('sqft', '').replace(',', '').strip()
    try:
        return int(float(s))
    except:
        return None

df['area_sqft'] = df['area_sqft'].apply(clean_area)

def clean_furnishing(val):
    if pd.isna(val):
        return 'Unfurnished'
    s = str(val).strip().lower()
    if 'semi' in s:
        return 'Semi-Furnished'
    elif 'furnish' in s:
        return 'Furnished'
    else:
        return 'Unfurnished'

df['furnishing_status'] = df['furnishing_status'].apply(clean_furnishing)
print(df['furnishing_status'].value_counts())


Semi-Furnished    414
Unfurnished       262
Furnished         174
Name: furnishing_status, dtype: int64


### 4. Normalizing Boolean Flags and Distances
Convert Yes/No variations (Y, N, 1, 0, yes, no) to standardized 'Yes' and 'No' strings. Parse strings with units ('500m', '1.2 km') into floating-point numbers representing kilometers.


In [1]:
bool_cols = ['parking', 'balcony', 'lift', 'power_backup', 'ac', 'wifi', 'food', 'laundry', 'gym', 'cctv', 'security', 'housekeeping', 'available']
def clean_boolean(val):
    if pd.isna(val):
        return 'No'
    s = str(val).strip().lower()
    if s in ['yes', 'y', '1', 'true']:
        return 'Yes'
    return 'No'

for col in bool_cols:
    df[col] = df[col].apply(clean_boolean)

def clean_distance(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    s = str(val).strip().lower().replace(' ', '')
    if s.endswith('km'):
        s = s[:-2]
    elif s.endswith('m'):
        try:
            return round(float(s[:-1]) / 1000.0, 2)
        except:
            return None
    try:
        return round(float(s), 2)
    except:
        return None

dist_cols = ['metro_distance_km', 'college_distance_km', 'office_distance_km', 'school_distance_km', 'hospital_distance_km']
for col in dist_cols:
    df[col] = df[col].apply(clean_distance)
df[dist_cols].head(3)


### 5. Deduplication & Outlier Removal
We inspect duplicate property IDs and drop extreme outlier rents: a small flat listed with a rent > ₹1L, and a huge flat listed for < ₹3k (data input errors).


In [1]:
print(f'Duplicates before dropping: {df.duplicated(subset=["property_id"]).sum()}')
df.drop_duplicates(subset=['property_id'], keep='first', inplace=True)

# Check outliers
outliers = df[((df['monthly_rent'] > 100000) & (df['area_sqft'] < 500)) | 
              ((df['monthly_rent'] < 3000) & (df['area_sqft'] > 1500))]
print(f'Dropping {len(outliers)} outliers:')
print(outliers[['property_id', 'locality', 'monthly_rent', 'area_sqft']])

df = df[~df['property_id'].isin(outliers['property_id'])]
df['rent_per_sqft'] = (df['monthly_rent'] / df['area_sqft']).round(2)

# Save cleaned output
df.to_csv('../data/processed/rental_listings_cleaned.csv', index=False)
print(f'Final cleaned shape: {df.shape}')


Duplicates before dropping: 10
Dropping 2 outliers:
   property_id         locality  monthly_rent  area_sqft
49    PROP_050  Shakarpur                150000        273
149   PROP_150  Greater Kailash            1200       1960
Final cleaned shape: (838, 44)
